## Step 2 — Object Detection and Bounding Box Selection

This notebook performs object detection on the cropped RGB images using a trained YOLO model. The goal is to identify tree trunks within each image and select the bounding box most likely to correspond to the target trunk.

The workflow performs the following tasks:

1. Run YOLO inference on all RGB images.
2. Load the corresponding depth image (`.npy`) for each RGB image.
3. Calculate the average depth value within each detected bounding box.
4. Select the bounding box that is both:
   - the tallest (largest vertical extent), and
   - closest to the camera (smallest average depth).
5. Save the selected bounding boxes in COCO annotation format (`JSON`).

These annotations are used in later processing steps to extract trunk regions and compute trunk diameter measurements.

An optional visualization step is also included to draw the selected bounding boxes on the images and save annotated copies for verification.

In [ ]:
# Import libraries
import json
import os
import numpy as np
import cv2
from ultralytics import YOLO

## Run YOLO and generate annotations

In [2]:
# Define JSON encoder for saving NumPy objects

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.float32):
            return float(obj)
        return json.JSONEncoder.default(self, obj)


# Load YOLO model
model = YOLO("path/to/yolo_weights.pt")

# Image and depth folders
image_folder = "path/to/image_folder"
depth_folder = "path/to/depth_folder"

# Run YOLO inference
results = model(image_folder)


# COCO format dictionary
coco_annotations = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 0, "name": "tree"}]
}

image_id = 0
annotation_id = 0


# Process detection results
for result in results:

    best_box = None
    best_height = 0
    min_depth = float("inf")

    depth_path = os.path.join(
        depth_folder,
        os.path.basename(result.path).replace(".png", ".npy")
    )

    if not os.path.exists(depth_path):
        print(f"Depth file not found: {depth_path}")
        continue

    depth_image = np.load(depth_path)

    for box in result.boxes.xyxy:

        x1, y1, x2, y2 = map(int, box)

        bbox_depth = depth_image[y1:y2, x1:x2]

        if bbox_depth.size == 0:
            continue

        avg_depth = np.mean(bbox_depth[bbox_depth > 0])

        box_height = y2 - y1

        if box_height > best_height and avg_depth < min_depth:
            best_box = box
            best_height = box_height
            min_depth = avg_depth

    if best_box is not None:

        best_box = np.array(best_box.cpu())

        annotation = {
            "id": annotation_id,
            "image_id": image_id,
            "category_id": 0,
            "bbox": [
                best_box[0],
                best_box[1],
                best_box[2] - best_box[0],
                best_box[3] - best_box[1],
            ],
            "area": float(
                (best_box[2] - best_box[0]) * (best_box[3] - best_box[1])
            ),
            "iscrowd": 0,
        }

        coco_annotations["annotations"].append(annotation)
        annotation_id += 1

    image_name = os.path.basename(result.path)

    image_width = result.orig_shape[1]
    image_height = result.orig_shape[0]

    coco_annotations["images"].append(
        {
            "id": image_id,
            "file_name": image_name,
            "width": image_width,
            "height": image_height,
        }
    )

    image_id += 1


# Save annotations
annotations_path = "path/to/yolo_annotations.json"

with open(annotations_path, "w") as json_file:
    json.dump(coco_annotations, json_file, cls=NumpyEncoder, indent=4)

print("Annotations saved successfully")



WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1589 D:\DepthImages\Blocks\Kaysville\091125\right\2025-09-11-13-41-16.0.png: 256x640 3 trees, 53.3ms
image 2/1589 D:\DepthImages\Blocks\Kaysville\091125\right\2025-09-11-13-41-17.0.png: 256x640 3 trees, 47.2ms
image 3/1589 D:\DepthImages\Blocks\Kaysville\091125\right\2025-09-11-13-41-18.0.png: 256x640 3 trees, 45.4ms
image 4/1589 D:\DepthImages\Blocks\Kaysville\091125\right\2025-09-11-13-41-19.0.png: 256x640 4 trees, 45.5ms
image 5/1589 D:\DepthI

C:\Users\a02393897\Anaconda3\envs\yolo\lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\a02393897\Anaconda3\envs\yolo\lib\site-packages\numpy\_core\_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Annotations saved successfully!


## Draw annotations on images

In [1]:
# Visualize annotations by drawing bounding boxes on images

image_folder = "path/to/image_folder"
annotations_path = "path/to/yolo_annotations.json"
output_folder = "path/to/annotated_images"

os.makedirs(output_folder, exist_ok=True)

with open(annotations_path, "r") as f:
    coco_data = json.load(f)


for image_info in coco_data["images"]:

    image_name = image_info["file_name"]
    image_path = os.path.join(image_folder, image_name)

    image = cv2.imread(image_path)

    if image is None:
        print(f"Warning: Could not load {image_path}")
        continue

    image_annotations = [
        ann
        for ann in coco_data["annotations"]
        if ann["image_id"] == image_info["id"]
    ]

    for ann in image_annotations:

        x, y, w, h = map(int, ann["bbox"])

        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(
            image,
            "Tree",
            (x, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2,
        )

    annotated_image_path = os.path.join(
        output_folder,
        image_name.replace(".png", "_annotation.png"),
    )

    cv2.imwrite(annotated_image_path, image)

    print(f"Saved: {annotated_image_path}")


print("Annotation images saved successfully")

Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-16.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-17.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-18.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-19.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-20.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-21.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-22.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-23.0_annotation.png
Saved: D:\DepthImages\Blocks\Kaysville\091125\left\annotated_images_v11\2025-09-11-13-41-24.0_annotation.png
Saved: D:\DepthImag